# video_vjepa2 — entropy field on a native-3D video ViT

Step E. ViT-B/16 and DINO-v2 (Step B) are still-image ViTs we apply
frame-by-frame, then volumetrically. V-JEPA 2 is the natural
control: a *native-3D* tubelet ViT trained on video, so its residual
stream at every layer is a 3-D scalar field on the (tubelet-time ×
spatial-y × spatial-x) grid by construction.

Two questions:
1. Does V-JEPA 2 also exhibit a sharp depth-wise phase transition in
   standardised per-token entropy, like the still-image ViTs?
2. Does the volumetric high-|∇H| component at any depth track real
   motion content in the input — or is it dominated by register-token-
   style edge artefacts (Darcet et al. 2024) and temporal positional
   structure injected by the model regardless of content?

Stimuli, ViT-L (24 layers, 1024 hidden, 32×16×16 token grid for
fpc=64 / 256² input):
- **Five real clips** from Big Buck Bunny, sampled at different
  timestamps to cover varied content (opening titles, character
  close-ups, object motion, scene cuts, camera pans). 64 frames at
  stride-2, 256². Real visual diversity, not a toy stimulus.
- **Constant-content control**: 64 identical copies of one frame.
  If V-JEPA 2 only encoded content, the resulting (gt, gy, gx)
  entropy volume should have ~zero variance along `t`.
- **Synthetic translating Gaussian blob** — kept ONLY as the
  sanity check for the volumetric tube-IoU metric, because it's the
  one stimulus with a known ground-truth trajectory in (t, y, x).
  It is *not* used in the phase / projection comparisons — too toy
  relative to V-JEPA 2's pretraining distribution.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from eris.extract import extract_entropy_volume
from eris.fields import gradient_3d
from eris.video import (
    blob_centres_to_patch,
    load_real_video,
    load_ssv2_clips,
    synthesise_translating_blob,
)
from eris.volumetric import extract_outlier_tubes, iou_3d, render_streamtubes_html

REPO = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
RESULTS = REPO / "results" / "video_vjepa2"
RESULTS.mkdir(parents=True, exist_ok=True)

ARCH = "vjepa2_l"
N_FRAMES = 64
IMG_SIZE = 256

## 1. Load the stimuli at 256²

Five real clips from Something-Something v2 — the canonical motion-
discrimination eval set, used to benchmark V-JEPA 2 in the original
paper. SSv2 clips are 2-6 s of human-object interaction; we take
the first 64 frames of each. Streamed from a parquet mirror so we
don't need the 19.5 GB full download.

In [2]:
N_SSV2_CLIPS = 5
ssv2_cache = REPO / "data" / "cache" / f"ssv2_{N_SSV2_CLIPS}clips_{N_FRAMES}f_{IMG_SIZE}.npz"
ssv2_clips, ssv2_metas = load_ssv2_clips(
    n_clips=N_SSV2_CLIPS, n_frames=N_FRAMES, size=IMG_SIZE,
    seed=42, cache_path=ssv2_cache,
)
print(f"ssv2: {ssv2_clips.shape}")
for i, m in enumerate(ssv2_metas):
    print(f"  clip {i}: native_frames={m['num_frames_total']} "
          f"fps={m['fps']:.1f} hxw={m['height']}x{m['width']}")

Resolving data files:   0%|          | 0/442 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/442 [00:00<?, ?it/s]

ssv2: (5, 64, 256, 256, 3)
  clip 0: native_frames=69 fps=12.0 hxw=240x360
  clip 1: native_frames=70 fps=11.8 hxw=240x427
  clip 2: native_frames=65 fps=12.0 hxw=240x360
  clip 3: native_frames=64 fps=12.0 hxw=240x427
  clip 4: native_frames=70 fps=12.0 hxw=240x427


### Constant-content control — does V-JEPA 2 invent temporal structure?

64 identical copies of the middle frame of SSv2 clip 0. If V-JEPA 2
only encoded content, the resulting (gt, gy, gx) entropy volume
should be ~constant along `t` (every tubelet sees the same image).
Any structure along `t` is an artefact of the model's positional
encoding rather than scene dynamics.

In [3]:
static_frame = ssv2_clips[0, N_FRAMES // 2]                       # (256, 256, 3)
const_frames = np.broadcast_to(static_frame, (N_FRAMES, IMG_SIZE, IMG_SIZE, 3)).copy()
print(f"const: {const_frames.shape}   "
      f"all frames identical: {np.array_equal(const_frames[0], const_frames[-1])}")

const: (64, 256, 256, 3)   all frames identical: True


### Synthetic translating blob — kept ONLY for the volumetric tube-IoU
sanity check in §6, since it's the one stimulus with a known ground-
truth trajectory in (t, y, x). Not used in the phase / projection
comparisons.

In [4]:
synth_frames, synth_meta = synthesise_translating_blob(
    n_frames=N_FRAMES, size=IMG_SIZE, sigma=14.0, amplitude=110.0,
    noise_std=12.0, seed=0,
)
print(f"synth: {synth_frames.shape}  blob crosses {synth_meta.centres_xy[0]} → "
      f"{synth_meta.centres_xy[-1]}")

synth: (64, 256, 256, 3)  blob crosses [ 25. 128.] → [230. 128.]


## 2. Run V-JEPA 2 → per-layer 3-D entropy volume per clip

In [5]:
def cached_volume(arch: str, name: str, frames: np.ndarray) -> np.ndarray:
    out_npz = RESULTS / f"{arch}_{name}_H_volume.npz"
    if out_npz.exists():
        return np.load(out_npz)["H"]
    vol, spec = extract_entropy_volume(
        arch, frames,
        progress=lambda L, n: print(f"  {name} L={L}/{n}", end="\r"),
    )
    np.savez_compressed(
        out_npz, H=vol,
        grid_size_3d=np.array(spec.grid_size_3d),
        n_layers=spec.n_layers,
    )
    print()
    return vol


H_ssv2 = [cached_volume(ARCH, f"ssv2{i}", ssv2_clips[i])
          for i in range(N_SSV2_CLIPS)]
H_const = cached_volume(ARCH, "const", const_frames)
H_synth = cached_volume(ARCH, "synth", synth_frames)
print(f"H_ssv2[0]: {H_ssv2[0].shape}   "
      f"H_const: {H_const.shape}   H_synth: {H_synth.shape}")

# Convenience labels for the multi-clip comparison.  Synth is dropped
# from this set; it returns only in §6 for the tube-IoU sanity check.
COMPARISON_VOLS = (
    [(f"ssv2_{i}", H_ssv2[i]) for i in range(N_SSV2_CLIPS)]
    + [("const", H_const)]
)

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

  ssv20 L=24/24


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

  ssv21 L=24/24


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

  ssv24 L=24/24


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

  const L=24/24


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

  synth L=24/24
H_ssv2[0]: (24, 32, 16, 16)   H_const: (24, 32, 16, 16)   H_synth: (24, 32, 16, 16)


## 3. Phase curves vs depth (mean, std, |∇H|, ΔH-std)

In [6]:
def phase_curves(H_vol: np.ndarray) -> dict[str, np.ndarray]:
    """H_vol of shape (n_layers, gt, gy, gx) → per-layer summary statistics."""
    n_L = H_vol.shape[0]
    flat = H_vol.reshape(n_L, -1)
    H_mean = flat.mean(axis=1)
    H_std = flat.std(axis=1)
    grad_mag = np.empty(n_L)
    for L in range(n_L):
        Ht, Hy, Hx = gradient_3d(H_vol[L])
        grad_mag[L] = np.sqrt(Ht ** 2 + Hy ** 2 + Hx ** 2).mean()
    dH_std = np.r_[0.0, np.abs(np.diff(H_std))]
    return {"H_mean": H_mean, "H_std": H_std,
            "grad_mag": grad_mag, "dH_std": dH_std}


curves = {name: phase_curves(vol) for name, vol in COMPARISON_VOLS}
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True)
metric_titles = [("H_mean", "H mean"),
                 ("H_std", "H std (within-layer spread)"),
                 ("grad_mag", "mean |∇₃H| per layer"),
                 ("dH_std", "ΔH-std (cross-layer change)")]
for ax, (key, title) in zip(axes, metric_titles):
    for name, c in curves.items():
        n_L = len(c[key])
        ls = "--" if name == "const" else "-"
        ax.plot(np.arange(1, n_L + 1), c[key], ls + "o", label=name, ms=3,
                lw=1.6 if name == "const" else 1.2,
                color="black" if name == "const" else None)
    ax.set_title(title); ax.set_xlabel("layer")
    ax.grid(True, alpha=0.3)
axes[0].legend(fontsize=8)
fig.suptitle(f"{ARCH} — per-layer entropy field summary, two video stimuli",
             y=1.01)
fig.tight_layout()
fig.savefig(RESULTS / "phase_curves.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"saved {RESULTS / 'phase_curves.png'}")

saved /home/varun/research/eris/results/video_vjepa2/phase_curves.png


### Identify the V-JEPA 2 transition layer per stimulus

In [7]:
trans_summary = []
for name, c in curves.items():
    L_argmax = int(np.argmax(c["dH_std"][1:]) + 1) + 1   # 1-based, ignore L=1 padding
    peak = float(c["dH_std"].max())
    median = float(np.median(c["dH_std"][1:]))
    sharpness = peak / max(median, 1e-9)
    trans_summary.append({"video": name, "L_trans": L_argmax,
                          "peak_dHstd": peak, "median_dHstd": median,
                          "peak_over_median": sharpness})
    print(f"{name}: L_trans={L_argmax}  peak/median ΔH-std = {sharpness:.2f}")
import pandas as pd
pd.DataFrame(trans_summary).to_csv(
    RESULTS / "transition_per_clip.csv", index=False)

ssv2_0: L_trans=24  peak/median ΔH-std = 5.65
ssv2_1: L_trans=24  peak/median ΔH-std = 4.84
ssv2_2: L_trans=24  peak/median ΔH-std = 2.83
ssv2_3: L_trans=24  peak/median ΔH-std = 4.30
ssv2_4: L_trans=3  peak/median ΔH-std = 6.38
const: L_trans=24  peak/median ΔH-std = 6.48


## 4. Per-tubelet transition stability

At each tubelet timestep `t` (out of 32), compute the per-`t` slice
`(n_layers, gy, gx)` and ask which layer maximises ΔH-std. If the
transition is video-stationary the per-tubelet `L_trans` should equal
the overall `L_trans` for ~all `t`, with std ≈ 0.

In [8]:
def per_tubelet_transition(H_vol: np.ndarray) -> np.ndarray:
    """Per tubelet t, transition layer = argmax_L |H_std(L,t) − H_std(L−1,t)|."""
    gt = H_vol.shape[1]
    out = np.empty(gt, dtype=int)
    for t in range(gt):
        slab = H_vol[:, t, :, :]
        std_per_L = slab.reshape(slab.shape[0], -1).std(axis=1)
        out[t] = int(np.argmax(np.abs(np.diff(std_per_L))) + 1) + 1   # 1-based
    return out


n_panels = len(COMPARISON_VOLS)
fig, axes = plt.subplots(1, n_panels, figsize=(3.4 * n_panels, 3.8),
                         sharey=True)
for ax, (name, vol) in zip(axes, COMPARISON_VOLS):
    ts = per_tubelet_transition(vol)
    ax.plot(np.arange(1, len(ts) + 1), ts, "o-", ms=4)
    ax.axhline(int(np.median(ts)), ls=":", c="gray",
               label=f"median = {int(np.median(ts))}")
    ax.set_xlabel("tubelet index t (1..32)")
    ax.set_ylabel("argmax ΔH-std layer")
    ax.set_title(f"{name}  (std = {ts.std():.2f})", fontsize=10)
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.suptitle(f"{ARCH} — per-tubelet transition layer stability", y=1.02)
fig.tight_layout()
fig.savefig(RESULTS / "transition_per_tubelet.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"saved {RESULTS / 'transition_per_tubelet.png'}")

saved /home/varun/research/eris/results/video_vjepa2/transition_per_tubelet.png


## 5. Layer × tubelet H_std heatmap — global view of the depth × time field

In [9]:
n_panels = len(COMPARISON_VOLS)
fig, axes = plt.subplots(n_panels, 1, figsize=(10, 2.4 * n_panels),
                         sharex=True)
for ax, (name, vol) in zip(axes, COMPARISON_VOLS):
    n_L, gt, gy, gx = vol.shape
    grid = vol.reshape(n_L, gt, -1).std(axis=2)            # (n_L, gt)
    im = ax.imshow(grid, aspect="auto", origin="lower",
                   extent=(1, gt, 1, n_L), cmap="magma")
    ax.set_ylabel("layer")
    ax.set_title(f"{name} — H_std(L, t)", fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="H_std")
axes[-1].set_xlabel("tubelet index t")
fig.suptitle(f"{ARCH} — per-(layer, tubelet) within-spatial spread",
             y=1.0)
fig.tight_layout()
fig.savefig(RESULTS / "layer_tubelet_grid.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"saved {RESULTS / 'layer_tubelet_grid.png'}")

saved /home/varun/research/eris/results/video_vjepa2/layer_tubelet_grid.png


## 5b. Const-input diagnostic — does V-JEPA 2 invent temporal structure?

H_const has identical input frames. Define
`Δt-std(L) = std over tubelet t of  spatial mean of  H_const[L, t, :, :]`.
If V-JEPA 2 strictly encoded content, Δt-std should be ~0 across all
layers; any non-zero values are a temporal positional / register
effect injected by the model. Compare against `Δt-std` for the synth
blob (where genuine temporal motion is present) and for the real
clip.

In [10]:
def t_std(H_vol: np.ndarray) -> np.ndarray:
    """For each layer, std over tubelet t of the spatial-mean H field."""
    return H_vol.mean(axis=(2, 3)).std(axis=1)


def xy_std(H_vol: np.ndarray) -> np.ndarray:
    """Per-layer mean spatial std (within each tubelet)."""
    return H_vol.std(axis=(2, 3)).mean(axis=1)


fig, ax = plt.subplots(figsize=(9, 5))
for name, vol in COMPARISON_VOLS:
    style = "k--o" if name == "const" else "-o"
    lw = 2.0 if name == "const" else 1.0
    ms = 5 if name == "const" else 3
    ax.plot(np.arange(1, vol.shape[0] + 1), t_std(vol), style,
            ms=ms, lw=lw, label=name)
ax.set_xlabel("layer")
ax.set_ylabel("std over tubelet t  of spatial-mean H")
ax.set_title("Temporal variance per layer  (5 SSv2 clips vs const).\n"
             "const should be ~0 if V-JEPA 2 only encodes content")
ax.grid(True, alpha=0.3); ax.legend(ncol=2, fontsize=9)
fig.tight_layout()
fig.savefig(RESULTS / "const_input_temporal_variance.png", dpi=140,
            bbox_inches="tight")
plt.close(fig)
print(f"saved {RESULTS / 'const_input_temporal_variance.png'}")

# Numeric ratio: const Δt-std vs the per-layer mean across the 5 SSv2 clips.
ssv2_t_std = np.stack([t_std(H_ssv2[i]) for i in range(N_SSV2_CLIPS)], axis=0)
ratios = pd.DataFrame({
    "L": np.arange(1, H_const.shape[0] + 1),
    "ssv2_t_std_mean": ssv2_t_std.mean(axis=0),
    "ssv2_t_std_std":  ssv2_t_std.std(axis=0),
    "const_t_std":     t_std(H_const),
})
ratios["const_over_ssv2mean"] = ratios["const_t_std"] / np.maximum(
    ratios["ssv2_t_std_mean"], 1e-12)
ratios.to_csv(RESULTS / "const_input_temporal_variance.csv", index=False)
print(ratios.round(4).to_string(index=False))

saved /home/varun/research/eris/results/video_vjepa2/const_input_temporal_variance.png
 L  ssv2_t_std_mean  ssv2_t_std_std  const_t_std  const_over_ssv2mean
 1           0.0247          0.0128       0.0008               0.0305
 2           0.0305          0.0199       0.0063               0.2053
 3           0.0307          0.0177       0.0425               1.3830
 4           0.0246          0.0117       0.0403               1.6368
 5           0.0164          0.0050       0.0588               3.5752
 6           0.0192          0.0083       0.0620               3.2225
 7           0.0195          0.0073       0.0532               2.7222
 8           0.0167          0.0090       0.0468               2.8016
 9           0.0176          0.0095       0.0570               3.2457
10           0.0177          0.0080       0.0514               2.9016
11           0.0177          0.0074       0.0471               2.6620
12           0.0193          0.0064       0.0561               2.9135
13 

## 5c. Volumetric projections — see the field's shape in (t, y, x)

For each clip + a few representative layers, project the H volume
and the gradient-magnitude volume |∇H| along each axis (max-project
for |∇H| to surface outlier tubes; mean-project for H to show the
bulk distribution). For the synth blob, the diagonal (t, x)
trajectory of the moving Gaussian should be visible in the
y-projection of |∇H|. For the const clip, all three projections
should show negligible temporal structure if V-JEPA 2 is content-
faithful.

In [11]:
def project_volume(vol: np.ndarray, op: str) -> dict[str, np.ndarray]:
    """3-D `(gt, gy, gx)` volume → three 2-D projections via ``op``.

    Returns ``{"t": (gy, gx), "y": (gt, gx), "x": (gt, gy)}``.
    `op` is one of "min" / "mean" / "max":
      - max: surfaces sparse high voxels (right tail);
      - min: surfaces sparse low voxels — entropy outliers in the
             ViT-feature literature live in this tail;
      - mean: bulk DC level — useful as a sanity reference.
    """
    fns = {"min": vol.min, "mean": vol.mean, "max": vol.max}
    if op not in fns:
        raise ValueError(f"unsupported projection op: {op}")
    fn = fns[op]
    return {
        "t": fn(axis=0),                           # collapse t → (gy, gx)
        "y": fn(axis=1),                           # collapse y → (gt, gx)
        "x": fn(axis=2),                           # collapse x → (gt, gy)
    }


def plot_projection_grid(
    vols_named: list[tuple[str, np.ndarray]], out_path: Path,
    layers_of_interest: list[int], title: str,
    cmap: str = "viridis",
    ops: tuple[str, str, str] = ("min", "mean", "max"),
) -> None:
    """For each (clip, layer): 3 axes × 3 ops = 9 panels in one row block.

    Layout: rows = clip × layer, cols grouped as 3 axes × 3 ops
    (so 9 cols total). Per-(clip, layer) row block lets the eye scan
    the same physical layer across t/y/x projections under all three
    projection operators side by side.
    """
    n_clips = len(vols_named)
    n_L = len(layers_of_interest)
    n_axes = 3
    n_ops = len(ops)
    n_rows = n_clips * n_L
    n_cols = n_axes * n_ops
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(2.0 * n_cols + 0.5, 2.4 * n_rows),
        squeeze=False,
    )
    axis_labels = ("t", "y", "x")
    for ci, (name, vol) in enumerate(vols_named):
        for li, L in enumerate(layers_of_interest):
            row = ci * n_L + li
            slab = vol[L - 1]                       # (gt, gy, gx)
            for ai, axlabel in enumerate(axis_labels):
                for oi, op in enumerate(ops):
                    proj = project_volume(slab, op=op)[axlabel]
                    col = ai * n_ops + oi
                    ax = axes[row, col]
                    im = ax.imshow(proj, cmap=cmap, aspect="auto")
                    ax.set_xticks([]); ax.set_yticks([])
                    if row == 0:
                        ax.set_title(f"axis={axlabel}\n{op}", fontsize=8)
                    if col == 0:
                        ax.set_ylabel(f"{name}\nL={L}", fontsize=9)
                    fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02)
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.close(fig)


# Pick layers spanning the depth: shallow / mid / late.
LAYERS_INTEREST = [4, 12, 24]

# Visualisation set: a small subset of SSv2 clips + const + synth.
# Five SSv2 clips would explode the figure; show 2 to keep it scannable.
PROJ_VOLS = [
    ("ssv2_0", H_ssv2[0]),
    ("ssv2_1", H_ssv2[1]),
    ("const",  H_const),
    ("synth",  H_synth),
]
plot_projection_grid(
    PROJ_VOLS,
    out_path=RESULTS / "H_projections.png",
    layers_of_interest=LAYERS_INTEREST,
    title=f"{ARCH} — per-layer H volume projections "
          f"(min / mean / max along each axis)",
    cmap="viridis",
)
print(f"saved {RESULTS / 'H_projections.png'}")

def gradmag_volume(vol: np.ndarray) -> np.ndarray:
    """Per-layer |∇₃H| volume of shape (n_layers, gt, gy, gx)."""
    n_L = vol.shape[0]
    out = np.empty_like(vol)
    for L in range(n_L):
        Ht, Hy, Hx = gradient_3d(vol[L])
        out[L] = np.sqrt(Ht ** 2 + Hy ** 2 + Hx ** 2)
    return out


grad_ssv2_0 = gradmag_volume(H_ssv2[0])
grad_ssv2_1 = gradmag_volume(H_ssv2[1])
grad_const = gradmag_volume(H_const)
grad_synth = gradmag_volume(H_synth)

plot_projection_grid(
    [("ssv2_0", grad_ssv2_0), ("ssv2_1", grad_ssv2_1),
     ("const",  grad_const),  ("synth",  grad_synth)],
    out_path=RESULTS / "gradH_projections.png",
    layers_of_interest=LAYERS_INTEREST,
    title=f"{ARCH} — per-layer |∇₃H| volume projections "
          f"(min / mean / max; synth row reveals motion as a diagonal "
          f"in y-axis cols at L=4)",
    cmap="inferno",
)
print(f"saved {RESULTS / 'gradH_projections.png'}")

saved /home/varun/research/eris/results/video_vjepa2/H_projections.png


saved /home/varun/research/eris/results/video_vjepa2/gradH_projections.png


## 6. Volumetric outlier-tube tracking on the synthetic blob

Step B's volumetric tracking on still-image ViTs failed (bundle IoU =
0 for ViT-B/16) due to register-token edge artefacts: 60–80 % of the
top-5% |∇H| voxels were on frame edges. Native-3D V-JEPA 2 has no
explicit register tokens; if the entropy field carries motion
structure, the largest connected component should now intersect the
ground-truth blob trajectory.

Build the GT tubelet mask in (gt=32, gy=16, gx=16). Each tubelet
covers 2 input frames; take the midpoint of those two frames'
centres.

In [12]:
gt_synth, gy_synth, gx_synth = H_synth.shape[1:]
centres = synth_meta.centres_xy                 # (n_frames=64, 2) in (cx_px, cy_px)
# average each pair of frames → 32 tubelet centres
tubelet_centres = centres.reshape(gt_synth, 2, 2).mean(axis=1)
patches = blob_centres_to_patch(
    tubelet_centres, image_size=IMG_SIZE, grid=(gy_synth, gx_synth)
)                                                # (gt, 2) cx, cy
gt_mask = np.zeros((gt_synth, gy_synth, gx_synth), dtype=bool)
for t in range(gt_synth):
    cxp, cyp = patches[t]
    gt_mask[t, cyp, cxp] = True
# 1-patch dilate (y, x only) to allow some jitter
import scipy.ndimage as ndi
struct2d = ndi.generate_binary_structure(2, 1)
for t in range(gt_synth):
    gt_mask[t] = ndi.binary_dilation(gt_mask[t], structure=struct2d, iterations=1)
print(f"GT tube voxels: {gt_mask.sum()} / {gt_mask.size}")

GT tube voxels: 160 / 8192


### Per-layer tube IoU on synth (largest connected component vs GT)

In [13]:
def tube_metrics_per_layer(H_vol: np.ndarray, gt_mask: np.ndarray,
                           top_pct: float = 5.0) -> np.ndarray:
    n_L = H_vol.shape[0]
    rows = []
    for L in range(n_L):
        Ht, Hy, Hx = gradient_3d(H_vol[L])
        gmag = np.sqrt(Ht ** 2 + Hy ** 2 + Hx ** 2)
        # diagnose register-token edge dominance
        edge = np.zeros_like(gmag, dtype=bool)
        edge[:, 0, :] = edge[:, -1, :] = True
        edge[:, :, 0] = edge[:, :, -1] = True
        thresh = float(np.percentile(gmag, 100.0 - top_pct))
        topmask = gmag >= thresh
        edge_frac = float((topmask & edge).sum()) / max(int(topmask.sum()), 1)
        # largest connected component IoU
        ext = extract_outlier_tubes(gmag, top_pct=top_pct, connectivity=2)
        rows.append({
            "L": L + 1,
            "top_thresh": thresh,
            "n_top_voxels": int(topmask.sum()),
            "edge_frac": edge_frac,
            "n_components": ext.n_components,
            "largest_size": ext.largest_size,
            "iou_largest_vs_gt": iou_3d(ext.largest_mask, gt_mask),
            "iou_topmask_vs_gt": iou_3d(topmask, gt_mask),
        })
    return rows


import pandas as pd
rows = tube_metrics_per_layer(H_synth, gt_mask, top_pct=5.0)
df = pd.DataFrame(rows)
df.to_csv(RESULTS / "tube_metrics_synth.csv", index=False)
print(df.round(4).to_string(index=False))

 L  top_thresh  n_top_voxels  edge_frac  n_components  largest_size  iou_largest_vs_gt  iou_topmask_vs_gt
 1      0.0824           410     0.1268            27           381             0.1560             0.1469
 2      0.1244           410     0.1146            22           389             0.2365             0.2258
 3      0.1222           410     0.1073            19           392             0.1974             0.1900
 4      0.1036           410     0.3317            98           271             0.0668             0.0517
 5      0.0909           410     0.3268            98           286             0.0445             0.0345
 6      0.0883           410     0.4317           116           235             0.0647             0.0440
 7      0.0753           410     0.6780           175            13             0.0000             0.0197
 8      0.0802           410     0.6561           166            26             0.0000             0.0018
 9      0.0698           410     0.8000       

### Plot tube-tracking results

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(df["L"], df["iou_largest_vs_gt"], "o-",
             label="largest CC vs GT")
axes[0].plot(df["L"], df["iou_topmask_vs_gt"], "s--",
             label="raw top-5% mask vs GT", ms=4)
axes[0].set_xlabel("layer"); axes[0].set_ylabel("IoU")
axes[0].set_title("3-D tube IoU vs ground-truth (synth)")
axes[0].grid(True, alpha=0.3); axes[0].legend()
axes[1].plot(df["L"], df["edge_frac"], "o-", color="firebrick")
axes[1].axhline(4/16, ls="--", c="gray",
                label="chance edge frac = 4/16 = 0.25")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("edge fraction")
axes[1].set_title("Top-5% |∇H| voxels on frame edge\n(register-token diagnostic)")
axes[1].grid(True, alpha=0.3); axes[1].legend()
axes[2].plot(df["L"], df["n_components"], "o-")
axes[2].set_xlabel("layer"); axes[2].set_ylabel("n connected components")
axes[2].set_title("Connected-component count")
axes[2].grid(True, alpha=0.3)
fig.suptitle(f"{ARCH} — volumetric tube tracking on synth blob "
             f"(GT trajectory in (t, y, x))", y=1.02)
fig.tight_layout()
fig.savefig(RESULTS / "tube_metrics_summary.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"saved {RESULTS / 'tube_metrics_summary.png'}")

saved /home/varun/research/eris/results/video_vjepa2/tube_metrics_summary.png


## 7. Best-IoU layer — animate the largest CC + the GT mask

In [15]:
best_L = int(df.loc[df["iou_largest_vs_gt"].idxmax(), "L"])
print(f"best IoU layer for synth tube tracking: L={best_L}, "
      f"IoU={df['iou_largest_vs_gt'].max():.3f}")

L_idx = best_L - 1
Ht, Hy, Hx = gradient_3d(H_synth[L_idx])
gmag = np.sqrt(Ht ** 2 + Hy ** 2 + Hx ** 2)
ext = extract_outlier_tubes(gmag, top_pct=5.0, connectivity=2)

# Render side-by-side animation: GT mask | predicted mask | overlay
import matplotlib.animation as anim
gt_, gy_, gx_ = gt_mask.shape
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
ims = [axes[i].imshow(np.zeros((gy_, gx_), dtype=float),
                       cmap="gray" if i == 0 else
                       ("inferno" if i == 1 else "viridis"),
                       vmin=0, vmax=1) for i in range(3)]
titles = ["GT tubelet mask", f"largest CC at L={best_L}",
          "GT (cyan) + pred (red)"]
for ax, t in zip(axes, titles):
    ax.set_title(t, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

def init():
    return ims

def update(t):
    overlay = np.zeros((gy_, gx_, 3), dtype=float)
    overlay[..., 1] = gt_mask[t].astype(float)         # GT in green/cyan
    overlay[..., 2] = gt_mask[t].astype(float)
    overlay[..., 0] = ext.largest_mask[t].astype(float)
    ims[0].set_data(gt_mask[t].astype(float))
    ims[1].set_data(ext.largest_mask[t].astype(float))
    axes[2].clear()
    axes[2].imshow(overlay)
    axes[2].set_xticks([]); axes[2].set_yticks([])
    axes[2].set_title("GT (cyan) + pred (red)", fontsize=10)
    fig.suptitle(f"{ARCH} synth blob, L={best_L}, t={t+1}/{gt_}", y=1.0)
    return ims

ani = anim.FuncAnimation(fig, update, frames=gt_, init_func=init,
                         interval=100, blit=False)
out_gif = RESULTS / f"tube_compare_L{best_L}.gif"
ani.save(out_gif, writer="pillow", fps=8)
plt.close(fig)
print(f"saved {out_gif}")

best IoU layer for synth tube tracking: L=2, IoU=0.236


saved /home/varun/research/eris/results/video_vjepa2/tube_compare_L2.gif


## 8. Per-frame H@best_L animation (visual on the input video)

In [16]:
def render_field_over_video(frames: np.ndarray, H_vol: np.ndarray, L_idx: int,
                            out_gif: Path, title: str) -> None:
    """Animate H_vol[L_idx, t, :, :] heatmap upsampled onto the input frame."""
    n_frames = frames.shape[0]
    gt_, gy_, gx_ = H_vol.shape[1:]
    H_layer = H_vol[L_idx]                 # (gt, gy, gx)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    img_ax, h_ax = axes
    ima = img_ax.imshow(frames[0])
    img_ax.set_xticks([]); img_ax.set_yticks([])
    img_ax.set_title("input frame", fontsize=10)
    imh = h_ax.imshow(H_layer[0], cmap="viridis",
                      vmin=H_layer.min(), vmax=H_layer.max())
    h_ax.set_xticks([]); h_ax.set_yticks([])
    h_ax.set_title(f"H @ L={L_idx+1} (per-tubelet)", fontsize=10)
    fig.colorbar(imh, ax=h_ax, fraction=0.04, pad=0.02, label="H (nats)")

    def update(frame_t):
        ima.set_data(frames[frame_t])
        # 2 frames per tubelet: int(frame_t / 2)
        tubelet = min(int(frame_t // 2), gt_ - 1)
        imh.set_data(H_layer[tubelet])
        fig.suptitle(f"{title}  frame {frame_t+1}/{n_frames}  "
                     f"tubelet {tubelet+1}/{gt_}", y=1.0)
        return ima, imh

    ani = anim.FuncAnimation(fig, update, frames=n_frames,
                             interval=80, blit=False)
    ani.save(out_gif, writer="pillow", fps=10)
    plt.close(fig)


render_field_over_video(synth_frames, H_synth, best_L - 1,
                        RESULTS / f"synth_H_L{best_L}.gif",
                        f"{ARCH} synth blob")
render_field_over_video(ssv2_clips[0], H_ssv2[0], best_L - 1,
                        RESULTS / f"ssv2_0_H_L{best_L}.gif",
                        f"{ARCH} ssv2 clip 0")
print(f"saved depth_x_time GIFs at L={best_L}")

saved depth_x_time GIFs at L=2


## 9. 3-D streamtubes interactive HTML at L_trans

In [17]:
L_idx = trans_summary[0]["L_trans"] - 1   # synth's transition layer
Ht_s, Hy_s, Hx_s = gradient_3d(H_synth[L_idx])
out_html = render_streamtubes_html(
    Hx=Hx_s, Hy=Hy_s, Ht=Ht_s,
    out_path=RESULTS / f"synth_streamtubes_L{L_idx+1}.html",
    title=f"{ARCH} synth blob L={L_idx+1}",
    starts=4, upsample_xy=4, sizeref=0.4,
)
print(f"saved {out_html}")

saved /home/varun/research/eris/results/video_vjepa2/synth_streamtubes_L24.html


## 10. Persist transition + tracking summary

CSV the headline numbers next to Step B's so the comparison is one
step away.

In [18]:
import json
summary = {
    "arch": ARCH,
    "model": "facebook/vjepa2-vitl-fpc64-256",
    "n_layers": int(H_const.shape[0]),
    "grid_size_3d": list(H_const.shape[1:]),
    "stimuli": {
        "ssv2_clips": [{"index": i, **m} for i, m in enumerate(ssv2_metas)],
        "const": "64 identical copies of ssv2_0[32]",
        "synth": "translating Gaussian blob; only used in §6 tube-IoU sanity",
    },
    "transition_per_video": trans_summary,
    "best_synth_tube_iou": {
        "L": best_L,
        "iou_largest_vs_gt": float(df["iou_largest_vs_gt"].max()),
        "edge_frac_at_best": float(df.loc[df["L"] == best_L, "edge_frac"].iloc[0]),
        "iou_topmask_vs_gt_at_best":
            float(df.loc[df["L"] == best_L, "iou_topmask_vs_gt"].iloc[0]),
    },
    "step_b_reference": {
        "vit_b16_synth_best_iou_largest": 0.000,
        "dinov2_synth_best_iou_largest_at_L6": 0.299,
        "vit_b16_synth_edge_frac_top5pct": "0.60–0.80",
    },
}
with (RESULTS / "summary.json").open("w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "arch": "vjepa2_l",
  "model": "facebook/vjepa2-vitl-fpc64-256",
  "n_layers": 24,
  "grid_size_3d": [
    32,
    16,
    16
  ],
  "stimuli": {
    "ssv2_clips": [
      {
        "index": 0,
        "num_frames_total": 69,
        "fps": 12.0,
        "height": 240,
        "width": 360,
        "n_frames_used": 64
      },
      {
        "index": 1,
        "num_frames_total": 70,
        "fps": 11.83231913455037,
        "height": 240,
        "width": 427,
        "n_frames_used": 64
      },
      {
        "index": 2,
        "num_frames_total": 65,
        "fps": 12.001477104874445,
        "height": 240,
        "width": 360,
        "n_frames_used": 64
      },
      {
        "index": 3,
        "num_frames_total": 64,
        "fps": 12.00075004687793,
        "height": 240,
        "width": 427,
        "n_frames_used": 64
      },
      {
        "index": 4,
        "num_frames_total": 70,
        "fps": 12.000685753471627,
        "height": 240,
        "width": 427